In [1]:
import pandas as pd
import os

In [2]:
df = pd.read_csv('main_result.csv')
df = df[df['gpt_model'] == 'gpt-4o']
df = df[df['n_self_alignment'].isin([5])]
df

,run_id,final_state,target_character,pe,gpt_model,branch_factor,exp_name,evaluator,total_iterations,n_self_alignment,...,Evaluation/reach_imp_perc,Evaluation/path_length,Evaluation/fn_imp_perc,Evaluation/fp_imp_perc,Evaluation/tn_imp_perc,Evaluation/tp_imp_perc,Evaluation/solvability,Evaluation/playability,score,Evaluation/naive_playability
168,3kk20v1f,finished,1,cot,gpt-4o,2,sa,hr,6,5,...,0.666667,27.846155,2.500000,0.4,0.000000,0.100000,0.200000,0.433333,0.033333,NaN
169,3kk20v1f,finished,1,cot,gpt-4o,2,sa,hr,6,5,...,0.233333,0.000000,3.000000,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,NaN
170,3kk20v1f,finished,1,cot,gpt-4o,2,sa,hr,6,5,...,0.433333,26.750000,2.733334,0.0,0.200000,0.066667,0.100000,0.266667,0.088889,NaN
171,3kk20v1f,finished,1,cot,gpt-4o,2,sa,hr,6,5,...,0.366667,26.285715,2.833333,0.0,0.133333,0.033333,0.066667,0.233333,0.055556,NaN
172,3kk20v1f,finished,1,cot,gpt-4o,2,sa,hr,6,5,...,0.700000,26.700001,1.800000,0.0,0.866667,0.333333,0.433333,0.666667,0.400000,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1075,s0bhdde9,finished,6,got,gpt-4o,2,def,hr,6,5,...,NaN,NaN,NaN,NaN,NaN,NaN,0.066667,0.000000,0.000000,0.066667
1076,s0bhdde9,finished,6,got,gpt-4o,2,def,hr,6,5,...,NaN,NaN,NaN,NaN,NaN,NaN,0.000000,0.000000,0.000000,0.000000
1077,s0bhdde9,finished,6,got,gpt-4o,2,def,hr,6,5,...,NaN,NaN,NaN,NaN,NaN,NaN,0.000000,0.000000,0.000000,0.000000
1078,s0bhdde9,finished,6,got,gpt-4o,2,def,hr,6,5,...,NaN,NaN,NaN,NaN,NaN,NaN,0.133333,0.000000,0.000000,0.133333


In [3]:
df.groupby(['gpt_model', 'pe', 'target_character', 'n_self_alignment']).agg({'Evaluation/llm_iteration': ['count']})

Evaluation/llm_iteration
                                                                   count
gpt_model pe  target_character n_self_alignment                         
gpt-4o    cot 1                5                                      30
              2                5                                      30
              5                5                                      30
              6                5                                      30
          got 1                5                                      30
              2                5                                      30
              5                5                                      30
              6                5                                      30
          tot 1                5                                      30
              2                5                                      30
              5                5                                      30
              6                5                                      30

In [4]:
# min-max normalization with 'score' column for target_character-wise
def min_max_normalize(df):
    df = df.copy()
    def normalize_group(sub_df):
        min_val = sub_df['score'].min()
        max_val = sub_df['score'].max()
        if max_val > min_val:
            sub_df['score'] = (sub_df['score'] - min_val) / (max_val - min_val)
        else:
            sub_df['score'] = 0.0  # 모든 값이 동일한 경우
        return sub_df
    return df.groupby('target_character').apply(normalize_group).reset_index(drop=True)

normalized = min_max_normalize(df)
normalized

/var/folders/x_/2lt9k5kn52q43m1_z0kp7tfm0000gn/T/ipykernel_50912/3838729616.py:12: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  return df.groupby('target_character').apply(normalize_group).reset_index(drop=True)


,run_id,final_state,target_character,pe,gpt_model,branch_factor,exp_name,evaluator,total_iterations,n_self_alignment,...,Evaluation/reach_imp_perc,Evaluation/path_length,Evaluation/fn_imp_perc,Evaluation/fp_imp_perc,Evaluation/tn_imp_perc,Evaluation/tp_imp_perc,Evaluation/solvability,Evaluation/playability,score,Evaluation/naive_playability
0,3kk20v1f,finished,1,cot,gpt-4o,2,sa,hr,6,5,...,0.666667,27.846155,2.500000,0.4,0.000000,0.100000,0.200000,0.433333,0.042857,NaN
1,3kk20v1f,finished,1,cot,gpt-4o,2,sa,hr,6,5,...,0.233333,0.000000,3.000000,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,NaN
2,3kk20v1f,finished,1,cot,gpt-4o,2,sa,hr,6,5,...,0.433333,26.750000,2.733334,0.0,0.200000,0.066667,0.100000,0.266667,0.114286,NaN
3,3kk20v1f,finished,1,cot,gpt-4o,2,sa,hr,6,5,...,0.366667,26.285715,2.833333,0.0,0.133333,0.033333,0.066667,0.233333,0.071429,NaN
4,3kk20v1f,finished,1,cot,gpt-4o,2,sa,hr,6,5,...,0.700000,26.700001,1.800000,0.0,0.866667,0.333333,0.433333,0.666667,0.514286,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
355,s0bhdde9,finished,6,got,gpt-4o,2,def,hr,6,5,...,NaN,NaN,NaN,NaN,NaN,NaN,0.066667,0.000000,0.000000,0.066667
356,s0bhdde9,finished,6,got,gpt-4o,2,def,hr,6,5,...,NaN,NaN,NaN,NaN,NaN,NaN,0.000000,0.000000,0.000000,0.000000
357,s0bhdde9,finished,6,got,gpt-4o,2,def,hr,6,5,...,NaN,NaN,NaN,NaN,NaN,NaN,0.000000,0.000000,0.000000,0.000000
358,s0bhdde9,finished,6,got,gpt-4o,2,def,hr,6,5,...,NaN,NaN,NaN,NaN,NaN,NaN,0.133333,0.000000,0.000000,0.133333


In [ ]:
# average the target_character
avg_df = normalized.groupby(['gpt_model', 'pe', 'n_self_alignment', 'Evaluation/llm_iteration', 'seed']).agg({'score': 'mean'}).reset_index()
avg_df['target_character'] = 'mean'
avg_df

In [6]:
iterwise_df = avg_df.groupby(['pe', 'Evaluation/llm_iteration']).agg({'score': ['mean']})
# sort with pe (cot, tot, got)
# melt with the Evaluation/llm_iteration
iterwise_df = iterwise_df.reset_index()
# Evaluation/llm_iteration to column
iterwise_df = iterwise_df.pivot(index='pe', columns='Evaluation/llm_iteration', values=('score', 'mean'))
iterwise_df.columns = [f'{col}' for col in iterwise_df.columns]
iterwise_df = iterwise_df.reindex(['cot', 'tot', 'got'], level=0)
iterwise_df

,1,2,3,4,5,6
pe,,,,,,
cot,0.162064,0.199550,0.370157,0.204837,0.161604,0.286833
tot,0.160119,0.191362,0.193661,0.343542,0.341826,0.361222
got,0.009292,0.337499,0.305011,0.190035,0.423717,0.427498


In [7]:
def df_to_latex_table(df: pd.DataFrame, caption="Performance between reasoning-based prompt engineering",
                      label="tab:reasoning_prompt", decimals=3) -> str:
    """
    Convert a DataFrame like iterwise_df to a LaTeX table.
    The maximum value in each row will be bolded.
    """
    df = df.copy()
    max_indices = df.astype(float).idxmax(axis=1)

    latex_rows = []
    for row_name, row in df.iterrows():
        row_entries = []
        for col in df.columns:
            val = round(row[col], decimals)
            if col == max_indices[row_name]:
                row_entries.append(f"\\textbf{{{val}}}")
            else:
                row_entries.append(f"{val:.{decimals}f}")
        latex_rows.append(f"{row_name} & " + " & ".join(row_entries) + " \\\\")

    # build LaTeX table with multi-column header
    latex = "\\begin{table}[!h]\n"
    latex += "\\centering\n"
    latex += f"\\caption{{{caption}}}\n"
    latex += f"\\label{{{label}}}\n"
    latex += "\\begin{tabular}{p{1.8cm}|" + "r" * df.shape[1] + "}\n"
    latex += "\\toprule\n"
    latex += "  \\textbf{PE} & \\multicolumn{" + str(df.shape[1]) + "}{c}{Iteration ($y_{i}$)} \\\\\n"
    latex += " & " + " & ".join([str(c) for c in df.columns]) + " \\\\\n"
    latex += "\\midrule\n"
    latex += "\n".join(latex_rows) + "\n"
    latex += "\\bottomrule\n"
    latex += "\\end{tabular}\n"
    latex += "\\end{table}"
    return latex

In [8]:
latex_code = df_to_latex_table(iterwise_df)
print(latex_code)

\begin{table}[!h]
\centering
\caption{Performance between reasoning-based prompt engineering}
\label{tab:reasoning_prompt}
\begin{tabular}{p{1.8cm}|rrrrrr}
\toprule
  \textbf{PE} & \multicolumn{6}{c}{Iteration ($y_{i}$)} \\
 & 1 & 2 & 3 & 4 & 5 & 6 \\
\midrule
cot & 0.162 & 0.200 & \textbf{0.37} & 0.205 & 0.162 & 0.287 \\
tot & 0.160 & 0.191 & 0.194 & 0.344 & 0.342 & \textbf{0.361} \\
got & 0.009 & 0.337 & 0.305 & 0.190 & 0.424 & \textbf{0.427} \\
\bottomrule
\end{tabular}
\end{table}


### Significance test between cot, tot, got for each iteration

In [18]:
last_iter_df = normalized[normalized['Evaluation/llm_iteration'] == 6]
last_iter_df

,run_id,final_state,target_character,pe,gpt_model,branch_factor,exp_name,evaluator,total_iterations,n_self_alignment,...,Evaluation/reach_imp_perc,Evaluation/path_length,Evaluation/fn_imp_perc,Evaluation/fp_imp_perc,Evaluation/tn_imp_perc,Evaluation/tp_imp_perc,Evaluation/solvability,Evaluation/playability,score,Evaluation/naive_playability
5,3kk20v1f,finished,1,cot,gpt-4o,2,sa,hr,6,5,...,0.800000,26.000002,1.733333,0.000000,0.933333,0.333333,0.466667,1.000000,0.542857,NaN
11,l59oo1h1,finished,1,cot,gpt-4o,2,sa,hr,6,5,...,0.633333,26.137932,2.566667,0.333333,0.000000,0.100000,0.166667,0.966667,0.042857,NaN
17,31kbu7of,finished,1,cot,gpt-4o,2,sa,hr,6,5,...,0.966667,27.500000,2.900000,0.066667,0.000000,0.033333,0.033333,0.266667,0.014286,NaN
23,ahfhtfmj,crashed,1,tot,gpt-4o,2,sa,hr,9,5,...,1.000000,26.000002,0.066667,2.000000,0.000000,0.933333,1.000000,1.000000,0.400000,NaN
29,wkddgr8z,crashed,1,tot,gpt-4o,2,sa,hr,9,5,...,0.966667,26.137932,0.833333,1.466667,0.000000,0.700000,0.733333,0.966667,0.300000,NaN
35,1wjo1jp0,crashed,1,tot,gpt-4o,2,sa,hr,9,5,...,0.266667,26.588236,2.033334,0.800000,0.000000,0.166667,0.400000,0.566667,0.071429,NaN
41,xacmuoqv,crashed,1,tot,gpt-4o,2,sa,hr,9,5,...,0.600000,26.000000,2.600000,0.333333,0.000000,0.066667,0.166667,0.966667,0.028571,NaN
47,3hcz0yw8,crashed,1,got,gpt-4o,2,sa,hr,9,5,...,0.100000,27.800001,2.733334,0.266667,0.000000,0.000000,0.133333,0.666667,0.000000,NaN
53,9ot6dizr,crashed,1,got,gpt-4o,2,sa,hr,9,5,...,1.000000,26.333334,0.100000,2.000000,0.000000,0.900000,1.000000,1.000000,0.385714,NaN
59,0m2oe7gc,crashed,1,got,gpt-4o,2,sa,hr,9,5,...,0.500000,27.000000,2.733334,0.100000,0.100000,0.066667,0.100000,0.133333,0.071429,NaN


In [19]:
from scipy.stats import ttest_rel

def welch_ttest(group1, group2):
    t_stat, p_value = ttest_rel(group1, group2)
    return p_value

# pairwise t-test between cot, tot, got
pe_pairs = [('cot', 'tot'), ('cot', 'got'), ('tot', 'got')]

results = []
for iter_num in sorted(last_iter_df['Evaluation/llm_iteration'].unique()):
    iter_data = last_iter_df[last_iter_df['Evaluation/llm_iteration'] == iter_num]
    for pe1, pe2 in pe_pairs:
        group1 = iter_data[iter_data['pe'] == pe1]['score']
        group2 = iter_data[iter_data['pe'] == pe2]['score']
        if len(group1) > 1 and len(group2) > 1:  # Ensure there are enough samples
            p_value = welch_ttest(group1, group2)
            results.append({
                'Iteration': iter_num,
                'PE1': pe1,
                'PE2': pe2,
                'p-value': p_value
            })
results_df = pd.DataFrame(results)
results_df


,Iteration,PE1,PE2,p-value
0,6,cot,tot,0.780683
1,6,cot,got,0.113730
2,6,tot,got,0.348129
